# NeuroGraph · Phase 6 · university intake
Settings: Accelerator **None**, Internet **On**. No inputs needed.
Fetches only the approved public pages in `configs/ingest_urls.yaml` (one request every 2 s). Raw pages stay in `/tmp` (they contain staff names); only redacted text is saved.

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install

In [ ]:
import os
R = "/kaggle/working/neurograph"
if not os.path.isdir(R):
    !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
!git -C {R} pull -q
!git -C {R} log --oneline -1
%cd {R}
!pip install -q -r requirements.txt -c /tmp/keep.txt
!pip install -q -e . -c /tmp/keep.txt

## 3. spaCy model for PII detection

In [ ]:
!python -m spacy download en_core_web_sm -q 2>&1 | tail -1

## 4. PII redaction tests

In [ ]:
!cd /kaggle/working/neurograph && python -m pytest tests/test_redact_pii.py -q -p no:cacheprovider 2>&1 | tail -3

## 5. Step 6.1: fetch -> clean -> PII redaction -> chunks

In [ ]:
!cd /kaggle/working/neurograph && python -m scripts.ingest_pages 2>&1 | grep -v -i warning